# SVM v2 — Option A (Calibration) + Option B (Zero-shot)

SVM RBF on 120-dim statistical features. Single model, 5 scenario evaluations.

In [1]:
import sys
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path: sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.utils import resample
from sklearn.metrics import accuracy_score

from config import RANDOM_SEED, MODELS_DIR
from src.experiment_runner import (
    get_splits, load_and_norm, split_cal_test,
    run_zero_shot, run_calibration, print_comparison,
    TEST_SUBJECTS, TRAIN_SUBJECTS,
)
from src.feature_extraction import extract_features_batch
from src.evaluation import measure_latency, print_latency

splits = get_splits()
print(f'Train subjects: {TRAIN_SUBJECTS}')
print(f'Test subjects:  {TEST_SUBJECTS}')

Train subjects: ['h0', 'h1', 'h10', 'h11', 'h12', 'h13', 'h14', 'h15', 'h18', 'h19', 'h2', 'h20', 'h21', 'h23', 'h25', 'h26', 'h27', 'h28', 'h29', 'h4', 'h5', 'h6', 'h8', 'h9']
Test subjects:  ['h7', 'h22', 'h3', 'h24', 'h16', 'h17']


## Train SVM

In [2]:
# Load + normalize training data
# Include S5 pre-fatigue in training for S5 evaluation
import pandas as pd
train_combined = pd.concat([splits['train_df'], splits['s5_train']])
X_train, y_train, norm_stats = load_and_norm(train_combined, verbose=True)
print(f'Train: {X_train.shape}')

# Extract features
print('Extracting features...')
F_train = extract_features_batch(X_train)
print(f'Features: {F_train.shape}')

# Subsample for SVM scalability
MAX_SVM = 50000
if len(F_train) > MAX_SVM:
    idx = resample(np.arange(len(y_train)), n_samples=MAX_SVM,
                   random_state=RANDOM_SEED, stratify=y_train)
    F_train_sub, y_train_sub = F_train[idx], y_train[idx]
    print(f'Subsampled to {MAX_SVM}')
else:
    F_train_sub, y_train_sub = F_train, y_train

scaler = StandardScaler().fit(F_train_sub)
F_train_sub = scaler.transform(F_train_sub)

print('Training SVM...')
svm = SVC(kernel='rbf', C=10, gamma='scale', random_state=RANDOM_SEED)
svm.fit(F_train_sub, y_train_sub)
print('Done.')

Loading windows: 100%|██████████| 5790/5790 [00:02<00:00, 2584.19it/s]


Train: (651972, 8, 50)
Extracting features...
Features: (651972, 120)
Subsampled to 50000
Training SVM...
Done.


## Option B — Zero-shot

In [3]:
def svm_predict(X):
    F = extract_features_batch(X)
    return svm.predict(scaler.transform(F))

print('Option B — Zero-shot:')
zero_results = run_zero_shot(svm_predict, splits, norm_stats, name='SVM')

Option B — Zero-shot:
  S1 zero-shot: 0.4772
  S2 zero-shot: 0.3959
  S3 zero-shot: 0.4033
  S4 zero-shot: 0.4938
  S5 zero-shot: 0.5717


## Option A — Calibration

For each test condition: retrain a fresh SVM on calibration features.

In [4]:
def svm_finetune(X_cal, y_cal):
    F_cal = scaler.transform(extract_features_batch(X_cal))
    svm_ft = SVC(kernel='rbf', C=10, gamma='scale', random_state=RANDOM_SEED)
    svm_ft.fit(F_cal, y_cal)
    def predict_ft(X):
        return svm_ft.predict(scaler.transform(extract_features_batch(X)))
    return predict_ft

print('Option A — Calibration:')
cal_results = run_calibration(svm_predict, svm_finetune, splits, norm_stats, name='SVM')

Option A — Calibration:
  S1 calibrated: 0.6293
  S2 calibrated: 0.6375
  S3 calibrated: 0.5896
  S4 calibrated: 0.6720
  S5 calibrated: 0.7443


## Latency

In [5]:
sample = X_train[:1]
def svm_single(x):
    f = extract_features_batch(x)
    return svm.predict(scaler.transform(f))
latency = measure_latency(svm_single, sample, n_runs=500)
print_latency(latency, 'SVM')


Latency — SVM
  Mean:   4.58 ms
  Median: 4.50 ms
  P95:    5.12 ms
  <300ms: ✓


## Results

In [6]:
print_comparison(zero_results, cal_results, name='SVM')


  SVM — RESULTS
Scenario        Zero-shot   Calibrated        Δ
-------------------------------------------------------
S1                47.72%       62.93%   +15.21%
S2                39.59%       63.75%   +24.16%
S3                40.33%       58.96%   +18.64%
S4                49.38%       67.20%   +17.82%
S5                57.17%       74.43%   +17.26%
